## 02 — Viewport Culling on a Live Map

We have `bbox_intersects`. Now we connect it to an actual map viewport.

ipyleaflet exposes the map's current bounds via `m.bounds` — the visible geographic rectangle. We read that rectangle, run the intersection test against every feature in the active LOD file, and send only the survivors to the renderer.

This notebook:
1. Reads the viewport from an ipyleaflet map
2. Applies the cull filter to a LOD file
3. Measures feature reduction at different zoom levels
4. Renders the culled result live

## Setup

In [1]:
import json
from pathlib import Path

lod_dir = Path("../../data/lod")

# Load the fine LOD file — a good test since it has the most features
with open(lod_dir / "railroads_fine.geojson") as f:
    fine = json.load(f)

features = fine["features"]
print(f"Fine LOD: {len(features):,} features")

Fine LOD: 25,413 features


In [2]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


def bbox_intersects(bbox_a, bbox_b):
    a_lon_min, a_lat_min, a_lon_max, a_lat_max = bbox_a
    b_lon_min, b_lat_min, b_lon_max, b_lat_max = bbox_b
    if a_lon_max < b_lon_min: return False
    if a_lon_min > b_lon_max: return False
    if a_lat_max < b_lat_min: return False
    if a_lat_min > b_lat_max: return False
    return True


def cull(features, viewport_bbox):
    """Return only the features whose bbox intersects viewport_bbox."""
    return [
        f for f in features
        if bbox_intersects(feature_bbox(f), viewport_bbox)
    ]

## Reading the Viewport from ipyleaflet

ipyleaflet's `m.bounds` returns the current visible bounds as:

```python
[[lat_min, lon_min], [lat_max, lon_max]]
```

Note the order: **latitude first, longitude second**. This is ipyleaflet's convention — the opposite of GeoJSON's `[lon, lat]`.

We need a helper to convert from ipyleaflet bounds to our `[lon_min, lat_min, lon_max, lat_max]` format.

In [3]:
def leaflet_bounds_to_bbox(bounds):
    """
    Convert ipyleaflet m.bounds format  [[lat_min, lon_min], [lat_max, lon_max]]
    to our bbox format                  [lon_min, lat_min, lon_max, lat_max]
    """
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]

## Static Culling — Measuring Reduction

Before building the live map, let's measure how effective culling is at several representative viewports.

In [4]:
viewports = [
    ("World (zoom 2)",          [-180, -90,  180,  90]),
    ("Europe (zoom 5)",         [ -10,  35,   40,  70]),
    ("France (zoom 7)",         [  -5,  42,   10,  52]),
    ("Paris region (zoom 10)",  [   1,  48,    4,  50]),
    ("Central Paris (zoom 13)", [ 2.3, 48.8, 2.4, 48.9]),
]

total = len(features)

print(f"{'Viewport':<26} {'Visible':>8} {'Culled':>8} {'% kept':>8}")
print("-" * 55)

for name, vp in viewports:
    visible = cull(features, vp)
    pct = len(visible) / total * 100
    culled = total - len(visible)
    print(f"{name:<26} {len(visible):>8,} {culled:>8,} {pct:>7.1f}%")

Viewport                    Visible   Culled   % kept
-------------------------------------------------------
World (zoom 2)               25,413        0   100.0%
Europe (zoom 5)              12,626   12,787    49.7%
France (zoom 7)               1,598   23,815     6.3%
Paris region (zoom 10)           51   25,362     0.2%
Central Paris (zoom 13)          11   25,402     0.0%


At city-level zoom, culling eliminates almost everything. At world zoom, nothing is culled — which is expected and acceptable because we would use the coarse LOD at that zoom anyway.

## Live Map with Viewport Culling

Now we wire it together. When the user stops panning or zooming, we:
1. Read `m.bounds`
2. Convert to our bbox format
3. Cull the features
4. Replace the GeoJSON layer with the culled result

In [5]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

m = Map(center=[48.5, 2.5], zoom=6)
status = widgets.Label(value="Pan or zoom to trigger culling")

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

def update_layer(*args):
    if not m.bounds:
        return
    viewport_bbox = leaflet_bounds_to_bbox(m.bounds)
    visible = cull(features, viewport_bbox)
    layer.data = {"type": "FeatureCollection", "features": visible}
    status.value = f"Viewport: {[round(v,2) for v in viewport_bbox]}  |  {len(visible):,} / {len(features):,} features visible"

m.observe(update_layer, names=["bounds"])
update_layer()  # Run once on load

widgets.VBox([m, status])

Pan around and zoom in. Watch the status line — the feature count drops dramatically as you zoom into a small region.

Notice also a limitation: each pan triggers a full linear scan of all features to recompute culling. With 25,000 features this is fast enough to feel acceptable. At 250,000 features it would lag. This is the motivation for Module 04 — a spatial grid index.

## Exercise A

The current map always uses the `fine` LOD file regardless of zoom. Modify the map so it uses `railroads_coarse.geojson` when zoom < 4 and `railroads_fine.geojson` when zoom >= 4.

This is a preview of Module 05 — just get it working here as a one-off.

In [6]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

lod_dir = Path("../../data/lod")

with open(lod_dir / "railroads_coarse.geojson") as f:
    coarse = json.load(f)

with open(lod_dir / "railroads_fine.geojson") as f:
    fine = json.load(f)


def leaflet_bounds_to_bbox(bounds):
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]


def choose_lod(zoom):
    if zoom < 4:
        return "coarse", coarse["features"]
    else:
        return "fine", fine["features"]


m = Map(center=[48.5, 2.5], zoom=3)

status = widgets.Label(value="Pan or zoom to update LOD")

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)

m.add(layer)


def update_layer(*args):
    if not m.bounds:
        return

    level_name, selected_features = choose_lod(m.zoom)

    viewport_bbox = leaflet_bounds_to_bbox(m.bounds)
    visible = cull(selected_features, viewport_bbox)

    layer.data = {
        "type": "FeatureCollection",
        "features": visible
    }

    status.value = (
        f"Zoom: {m.zoom} | LOD: {level_name} | "
        f"{len(visible):,} / {len(selected_features):,} features visible"
    )


m.observe(update_layer, names=["bounds"])
m.observe(update_layer, names=["zoom"])

update_layer()

widgets.VBox([m, status])

## Exercise B

The culling function runs a **linear scan** — it checks every feature in order. For the fine LOD with ~20,000 features, measure how long a single cull call takes using `time.perf_counter()`.

Then calculate: if the user pans 10 times per second, how much CPU time does culling consume per second? Is this a problem?

In [7]:
import time

# Example viewport
viewport = [-10, 35, 30, 60]

# Time one cull() call
t0 = time.perf_counter()

visible = cull(features, viewport)

elapsed = time.perf_counter() - t0

print(f"Single cull() call time: {elapsed:.6f} seconds")
print(f"Visible features: {len(visible):,}")

# CPU cost if user pans 10 times per second
cpu_time_per_second = elapsed * 10

print(f"\nCPU time at 10 pans/sec: {cpu_time_per_second:.6f} seconds per second")

# Interpretation
if cpu_time_per_second < 1:
    print("This is probably manageable.")
else:
    print("This could become a performance problem.")

Single cull() call time: 0.078633 seconds
Visible features: 9,836

CPU time at 10 pans/sec: 0.786334 seconds per second
This is probably manageable.


A linear scan becomes more expensive as the dataset grows because every cull call checks every feature. At 10 pans per second, the total CPU time can add up quickly, especially with larger datasets or more detailed geometry. This is why spatial indexes are important — they avoid scanning all features every time the viewport changes.

## Check Your Understanding

Our culling test uses **bounding boxes of the features**, not the actual line geometry.

A long diagonal railroad that runs from the bottom-left to the top-right corner of Europe has a bounding box that covers the entire continent. When the user is zoomed into Portugal, this feature passes the cull test — even though the actual track is nowhere near Portugal.

Is this a flaw in the algorithm? What would a more precise solution require — and is it worth it for a LOD rendering pipeline?

---

This is a limitation, but not a serious flaw. Bounding boxes are meant to be a fast first-pass filter, so they can allow some false positives like a long diagonal railroad whose box overlaps Portugal. A more precise solution would test the actual line geometry against the viewport, but that costs more time, so for an LOD rendering pipeline the bounding box approach is usually acceptable.

## Next

In [Module 04 — Spatial Grid Index](../04-Spatial_Grid_Index/README.md), we replace the linear scan with a grid-based index that makes culling faster as the dataset grows.